# 🔎 Baseball Lab 10: My Own Investigation
This is the last lab, and it's different: **you are the boss today.**

Baseball fans argue all the time. "Ruth was the greatest!" "No way, Aaron hit more!"
Most people argue with loud voices. You can now argue with **proof**: real numbers and a real chart.

Every investigation has the same 4 steps:

| step | what you do | what you use |
|---|---|---|
| 1. **Question** | write down exactly what you want to know | variables (Lab 1) |
| 2. **Numbers** | get the facts from the data | a machine and a dictionary (Labs 6 and 7) |
| 3. **Dig deeper** | look closer: one number never tells the whole story | a loop with a counter (Lab 5) |
| 4. **Picture and answer** | draw the chart, write the answer in your own words | charts (Lab 9) |

First we do one investigation **together**, as an example. Then you change the first box and it becomes **yours**.

It takes about **40 minutes**. Good plan: do the example one day (Parts 1 to 4), and your own investigation another day.

There are **no blanks** today. Click a gray box and press **Shift + Enter** to run it, as always.

Load the data with the hidden box (click ▶ and wait about 20 seconds).

In [ ]:
#@title ⚾ Load the MLB data, 1871 to today (click ▶ and wait about 20 seconds)
import datetime
import unicodedata

import pandas as pd
import requests

LAHMAN = "https://raw.githubusercontent.com/daviddalpiaz/pylahman/main/data-raw/"
MLB = "https://statsapi.mlb.com/api/v1/"
STATS = ["G", "AB", "H", "HR", "RBI", "SB", "BB", "SO"]


def plain(text):
    # "Acuña" -> "Acuna": kids type names on an English keyboard
    return unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()


def short(text):
    # "Acuña Jr." -> "acu": the start of a last name, to match players across the two sources
    return plain(text).lower()[:3]


# History: the Lahman Baseball Database (SABR, CC BY-SA 3.0). One row per player, season and team.
batting = pd.read_csv(LAHMAN + "Batting.csv")
people = pd.read_csv(LAHMAN + "People.csv")
teams = pd.read_csv(LAHMAN + "Teams.csv")[["yearID", "teamID", "name"]].rename(columns={"name": "team"})
batting = batting.merge(teams, on=["yearID", "teamID"], how="left")

# A player traded during a season has a row for each team: add them up and keep his last team.
per_season = batting.groupby(["playerID", "yearID"], sort=False)
history = per_season[STATS].sum(min_count=1).join(per_season["team"].last()).reset_index()
people["name"] = people["nameFirst"] + " " + people["nameLast"]
history = history.merge(people[["playerID", "name"]], on="playerID").rename(columns={"yearID": "season"})

# The seasons the history book doesn't have yet come live from MLB (statsapi.mlb.com).
lahman_ids = {
    (int(p.birthYear), int(p.birthMonth), int(p.birthDay), short(p.nameLast)): p.playerID
    for p in people.dropna(subset=["birthYear", "birthMonth", "birthDay"]).itertuples()
}
mlb_names, live = {}, []
try:
    for season in range(int(history["season"].max()) + 1, datetime.date.today().year + 1):
        url = MLB + f"stats?stats=season&group=hitting&season={season}&sportId=1&playerPool=ALL&limit=5000"
        found = requests.get(url, timeout=60).json().get("stats", [])
        hitters = found[0]["splits"] if found else []
        if not hitters:
            continue
        roster = requests.get(MLB + f"sports/1/players?season={season}", timeout=60).json()["people"]
        born = {p["id"]: p.get("birthDate", "") for p in roster}
        for h in hitters:
            p, s = h["player"], h["stat"]
            y, m, d = (born.get(p["id"]) or "0-0-0").split("-")
            player_id = lahman_ids.get((int(y), int(m), int(d), short(p["lastName"])), f"mlb{p['id']}")
            mlb_names[player_id] = p["fullName"]
            live.append({
                "playerID": player_id, "season": season, "team": h.get("team", {}).get("name"), "name": p["fullName"],
                "G": s["gamesPlayed"], "AB": s["atBats"], "H": s["hits"], "HR": s["homeRuns"], "RBI": s["rbi"],
                "SB": s["stolenBases"], "BB": s["baseOnBalls"], "SO": s["strikeOuts"],
            })
except (requests.RequestException, ValueError, KeyError) as problem:
    live = []
    print("MLB's live stats didn't answer, so the newest season is missing today. Everything else works.", problem)

stats = pd.concat([history, pd.DataFrame(live)], ignore_index=True)
# Players active today get the name MLB uses now on their whole career ("Vladimir Guerrero Jr."), without accents.
stats["name"] = stats["playerID"].map(mlb_names).fillna(stats["name"]).map(plain)
# Whole numbers (114, not 114.0); a stat nobody counted back then stays empty: <NA>.
stats[STATS] = stats[STATS].astype("Int64")
stats = stats.sort_values(["playerID", "season"], ignore_index=True)
stats = stats[["name", "season", "team"] + STATS + ["playerID"]]
print("Ready!", len(stats), "rows, seasons", stats["season"].min(), "to", stats["season"].max())

---
## Part 1. The question ❓
A good question has **two players** and **one thing to count**. "Who is the best?" is too fuzzy: best at what?
"Who hit more home runs?" is a question data can answer.

Here is our example. This box is the **control room** of the whole lab: every box below uses these five variables.

In [ ]:
question = "Who was the bigger home run hitter: Babe Ruth or Hank Aaron?"
player1 = "Babe Ruth"
player2 = "Hank Aaron"
stat = "HR"   # what we count: "HR", "SB", "H", "BB", "SO" or "G"
big = 40      # how many in one season counts as a BIG season

print("My question:", question)

| variable | what it holds |
|---|---|
| `question` | your question, as text. It will be printed on your final report |
| `player1`, `player2` | the two players, spelled exactly like in the data |
| `stat` | the column to count: `"HR"` home runs, `"SB"` stolen bases, `"H"` hits, `"BB"` walks, `"SO"` strikeouts, `"G"` games |
| `big` | what counts as a **big** season for that stat. For home runs, 40 is a great season |

Before going on, a careful investigator checks that Python can **find** both players.
One wrong letter and it finds nothing (Lab 2).

In [ ]:
for name in [player1, player2]:
    rows = stats[stats["name"] == name]
    if len(rows) == 0:
        print("⚠️ I can't find", name, "- check the spelling and the capital letters")
    else:
        print("Found", name, ":", len(rows), "seasons, from", rows["season"].min(), "to", rows["season"].max())

A `for` loop (Lab 5) goes through the two names. For each one, `len(rows)` counts his seasons (Lab 2),
and an `if` (Lab 3) decides what to say: zero rows means the name is wrong.

Both found: Ruth played from 1914 to 1935, Aaron from 1954 to 1976. They never played each other.
Only data can compare them!

---
## Part 2. The numbers 🔢
Now the facts. This machine (Lab 6) builds a **player card** (Lab 7): a dictionary with four facts about one player.

In [ ]:
def card(name, stat):
    rows = stats[stats["name"] == name].dropna(subset=[stat])
    return {
        "seasons": len(rows),
        "total": int(rows[stat].sum()),
        "best": int(rows[stat].max()),
        "average": round(rows[stat].mean()),
    }


card1 = card(player1, stat)
card2 = card(player2, stat)

print(player1, card1)
print(player2, card2)

| line | what it does |
|---|---|
| `def card(name, stat):` | a machine with two inputs: a player and a stat |
| `stats[stats["name"] == name]` | keep only this player's rows (Lab 2) |
| `.dropna(subset=[stat])` | throw out seasons where nobody wrote that stat down (Lab 1). Long ago, strikeouts and stolen bases weren't always counted |
| `return { ... }` | hand back a dictionary: four keys, four values |
| `"total"`: `.sum()` | all his seasons added up |
| `"best"`: `.max()` | his biggest single season |
| `"average"`: `.mean()` | a normal season for him: add up all seasons and share them out equally (Lab 2). `round( )` makes it a whole number |
| `int( )` | makes it a plain whole number, so it prints nicely (Lab 8) |

Now let's line the two cards up, fact by fact, and let Python name the leader:

In [ ]:
for key in ["seasons", "total", "best", "average"]:
    print(key, "->", player1, card1[key], "|", player2, card2[key])

if card1["total"] > card2["total"]:
    print("Bigger total:", player1)
elif card1["total"] == card2["total"]:
    print("The totals are a tie!")
else:
    print("Bigger total:", player2)

The loop walks through the four keys, and `card1[key]` looks each one up (Lab 7).

So is the case closed? Hank Aaron has the bigger total: **755** home runs to Ruth's **714**.

Not so fast! Look at the other lines. Aaron played one more season. And Ruth's **best** season (60) is much bigger than Aaron's (47).
A good investigator never stops at one number.

---
## Part 3. Dig deeper 🔍
Let's ask a sharper question: **how many BIG seasons did each one have?**
For home runs we said `big = 40`. This machine counts them with a loop and a counter, just like the 30-homer seasons in Lab 5.

In [ ]:
def big_seasons(name, stat, big):
    rows = stats[stats["name"] == name].dropna(subset=[stat])
    count = 0
    for number in list(rows[stat]):
        if number >= big:
            count = count + 1
    return count


count1 = big_seasons(player1, stat, big)
count2 = big_seasons(player2, stat, big)

print(player1, "- seasons with", big, "or more:", count1)
print(player2, "- seasons with", big, "or more:", count2)

Inside the machine:
- `count = 0` starts the counter at zero
- `list(rows[stat])` turns the column into a list of numbers, one per season (Lab 4)
- the loop looks at each season's number; `if number >= big` asks "was it a big one?" (Lab 3)
- `count = count + 1` adds one to the counter each time the answer is yes

Now the story changes: Ruth had **11** seasons with 40 or more home runs, Aaron had **8**.

Which seasons were their very best? Sorting (Lab 2) shows each player's top 3:

In [ ]:
for name in [player1, player2]:
    rows = stats[stats["name"] == name].dropna(subset=[stat])
    top3 = rows.sort_values(stat, ascending=False).head(3)
    print(top3[["name", "season", "team", stat]])
    print()

(The empty `print()` just leaves a blank line between the two tables.)

Ruth's top 3 are 60, 59 and 54. Aaron's are 47, 45, and 44. Ruth's **third-best** season beats Aaron's **best**!

---
## Part 4. The picture and the answer 📈
Numbers are the proof. A chart lets everybody **see** it. This is the side-by-side chart from Lab 9:
each player's career, counted from his 1st season.

In [ ]:
import matplotlib.pyplot as plt

words = {"HR": "Home runs", "SB": "Stolen bases", "H": "Hits", "BB": "Walks", "SO": "Strikeouts", "G": "Games"}

for name in [player1, player2]:
    rows = stats[stats["name"] == name].dropna(subset=[stat])
    number = range(1, len(rows) + 1)
    plt.plot(number, rows[stat], marker="o", label=name)

plt.legend()
plt.title(player1 + " and " + player2 + ": " + words[stat])
plt.xlabel("Season of his career   (1 = his first season)")
plt.ylabel(words[stat] + "   (higher = more)")
plt.show()

Look at the two shapes:
- **Ruth** starts at the bottom (he was a pitcher for his first years), then climbs two huge mountains, far above Aaron's line.
- **Aaron** starts hitting right away and stays between 24 and 47 for almost twenty seasons. No mountain, but a long, high road.

### The answer
The last step is the most important one, and Python can't do it for you: **say what you found, in your own words.**
A good answer says **who**, and gives the **numbers that prove it**.

In [ ]:
answer = "Hank Aaron hit more home runs in total (755 to 714). But Babe Ruth had more big seasons (11 to 8) and the best season (60). Aaron was great for longer, Ruth was greater at his best."

print("QUESTION:", question)
print()
print(player1, card1, "- big seasons:", count1)
print(player2, card2, "- big seasons:", count2)
print()
print("MY ANSWER:", answer)

That's a full investigation: a question, the numbers, a closer look, a picture and an answer.
Notice that the answer isn't just one name. Real answers often sound like "**it depends** what you mean by bigger". That's not cheating: that's being honest.

---
## 🔎 Now it's YOUR investigation
1. Scroll back up to the **control room** box in Part 1 (the one that starts with `question =`).
2. Change the five lines: your question, your two players, your stat, and what counts as a big season.
3. Run **every box again**, from the control room down, one by one. Read what each one shows.
4. In the last box, delete our answer and **write your own** between the quotes.
5. Show the chart and the answer to mom or dad, and explain how you know.

**Need an idea?** Try one of these (copy the names exactly):

| question | `player1` | `player2` | `stat` | `big` |
|---|---|---|---|---|
| Who was the bigger base stealer? | `"Rickey Henderson"` | `"Lou Brock"` | `"SB"` | `50` |
| Who was the bigger hit machine? | `"Ichiro Suzuki"` | `"Derek Jeter"` | `"H"` | `200` |
| Who is the bigger home run hitter today? | `"Aaron Judge"` | `"Shohei Ohtani"` | `"HR"` | `40` |
| Who walked more? | `"Barry Bonds"` | `"Ted Williams"` | `"BB"` | `100` |
| Who struck out more? | `"Reggie Jackson"` | `"Jim Thome"` | `"SO"` | `150` |

Or even better: use **your own** two favorite players.

**If something goes wrong:**
- the second box says ⚠️: fix the spelling of that name, then go on
- a red `TypeError` in Part 2: a name is still misspelled, so Python has no seasons to add up
- a red `KeyError`: `stat` must be one of the six codes, in capital letters and in quotes

Here is an empty box for anything else you want to try. It's yours.

In [ ]:
# your own code here

---
### 🏆 You did it, Investigator!
You finished all 10 labs. Look at what you can do now:
- keep things in **variables**, and do math with them
- make decisions with `if`, `elif` and `else`
- keep players in **lists** and facts in **dictionaries**
- repeat things with `for` and `while` loops
- build your own machines with `def` and `return`
- ask questions with `input( )` and pick by surprise with `random`
- find, sort and add up real stats in a table
- draw charts with clear words on them
- and the biggest one: **ask your own question and prove the answer with data**

That last one is what real sports analysts do for a living. The front office of every MLB team is full of people
doing exactly what you just did.

Keep investigating! ⚾